# Request 4 — Q3 KPI Snapshot (CPGO / Thursday board pack)
## Notebook build plan: `notebooks/q3 kpi snapshot.ipynb`

Verified facts from the actual data (checked 2026-11-23):
- Sheets: `users`, `transactions`, `loans`, `plus_subscriptions` (+ README, cvm, marketing — not needed here)
- `transactions.status` ∈ {success, failed}; `txn_type` ∈ {deposit, bill_payment, card_purchase, p2p_send, withdrawal}
- **577 duplicate `txn_id`s** in transactions → must dedup
- 0 orphan `user_id`s in transactions; 0 duplicate users; **17 users with test/demo/fake names**
- Loans have explicit `fee_egp` column (≈8% of principal) and `last_repayment_date`; fully-repaid example: 1500 + 120 = 1620 repaid ✔
- Plus fee is constant 49 EGP/month

---

## Cell 0 — Setup & constants (markdown + code)
```python
EXTRACT_TS   = pd.Timestamp("2026-11-05")          # as-of date for stock metrics
Q3_START_CAIRO = pd.Timestamp("2026-07-01 00:00")  # Cairo business time
Q3_END_CAIRO   = pd.Timestamp("2026-10-01 00:00")  # exclusive
CAIRO_OFFSET   = pd.Timedelta(hours=2)             # UTC -> Cairo (state assumption: no DST)
HEADLINE_CLAIM_EGP = 59_000_000
```
- Load all 4 sheets; parse dates once (`pd.to_datetime`).
- Print dtypes + row counts as a provenance cell.

## Cell 1 — Data-quality gate (feeds triage log too)
Do this BEFORE any metric. Log every finding as issue / detected-by / rows affected / handling / impact.
1. **Duplicate txn_id**: `t[t.txn_id.duplicated(keep=False)]` — inspect whether dupes are exact copies or same-id-different-row. Handling: drop exact duplicates (`drop_duplicates(subset='txn_id')` after sorting by txn_ts); report how many fall inside Q3 deposits (this is one candidate cause of the 59M gap).
2. **Test/internal users**: flag `full_name ~* 'test|demo|fake'` (17 users found). Also check phone/email patterns if columns existed (they don't here). Build `bad_user_ids` set and exclude consistently from **all four tables**.
3. **Sanity checks**: negative/zero/NaN amounts, NaN status, txn before signup (`txn_ts < signup_ts`), future timestamps > extract date.
4. Output: `dq_summary` DataFrame printed at the end — this doubles as the Part 2 log entries for this request.

## Cell 2 — Timezone conversion (one canonical step)
- Add `txn_ts_cairo = txn_ts + CAIRO_OFFSET` (and equivalents for `signup_ts`, `kyc_verified_ts`).
- Define quarter masks ONCE on cairo columns: `is_q3 = (ts_cairo >= Q3_START_CAIRO) & (ts_cairo < Q3_END_CAIRO)`.
- Keep both UTC and Cairo sums side by side — boundary-day difference is another candidate explanation for the headline gap.

---

## Cell 3 — ⭐ HEADLINE: Q3 deposit volume (the thing he actually asked to confirm)
Exactly your plan #1: clean → convert tz → filter → sum.
```python
dep = (txns_clean
       .query("txn_type == 'deposit' and status == 'success'")
       .loc[lambda d: d.is_q3])
headline = dep.amount_egp.sum()
```
Then build a **bridge/waterfall table** from 59M to your number, testing each hypothesis explicitly:
| Step | Sum (EGP) | Δ |
|---|---|---|
| Naive: all rows, type=deposit, any status, UTC window, dupes kept, test users kept | ? | vs 59M |
| + success only | ? | failed deposits counted? |
| + dedup txn_id | ? | duplicates counted twice? |
| + Cairo timezone | ? | boundary shift? |
| − test users | ? | internal wallets? |
| (check) other txn types included? p2p/deposit+withdrawal mixed? | ? | wrong filter? |
One of these steps should land near 59M → that identifies *their* error. State the correct figure + cause in one sentence.

## Cell 4 — Signups
- `users.signup_ts_cairo` in Q3 → count. Split by month (Jul/Aug/Sep) for trend context.
- Exclusions applied: test users. Note assumption: signup = `signup_ts` (install data isn't in users table).

## Cell 5 — Activation rate (README definition — do NOT invent one)
- Activated = `kyc_verified_ts` not null **AND** ≥1 successful **deposit** within 7 days (168h) of `signup_ts`.
- Cohort discipline: denominator = Q3 signups only. Numerator = Q3 signups meeting the definition (KYC and deposit anchored to their own signup clock, can occur after Sep 30 but ≤ signup+7d; safe since extract Nov 5 ≫ Sep 30+7d).
- Report: `activated / q3_signups` as %, plus monthly breakdown. State the two sub-decisions you made: (a) does KYC need to precede the deposit? (recommend: no, just both within window), (b) 7 days = calendar or rolling 168h (recommend: rolling).

## Cell 6 — MAU (definition is missing → define it yourself, loudly)
- Chosen definition: **distinct users with ≥1 successful transaction of any type in the Cairo calendar month.**
- Group by month × user, `count >= 1`, then count users per month → 3 numbers (Jul, Aug, Sep) + avg.
- One markdown line acknowledging alternatives (any-status incl. failed; activity = deposit-only; 7-day-active-session logic) and why you chose the simple standard fintech definition. Sensitivity: also print deposit-only MAU so CPGO knows the number moves with definition.

## Cell 7 — Loans disbursed
- From `loans`: `disbursed_date` in Q3 (date column — check whether tz applies; likely pure date).
- Report **both**: count of loans and total `principal_egp` (cash out). Exclude bad users.

## Cell 8 — Loan default rate DPD30 (your plan #2 — mind the censoring trap)
Per loan:
```python
expected_total = principal_egp + fee_egp            # use given fee column, don't recompute 8%
fully_repaid   = amount_repaid_egp >= expected_total - 0.01
default        = (~fully_repaid) & (EXTRACT_TS >= due_date + pd.Timedelta(days=30))
observable     = EXTRACT_TS >= due_date + pd.Timedelta(days=30)   # matured enough to KNOW
```
- Rate = defaults / observable loans. **Denominator = observable (matured) loans, NOT all Q3 loans** — mid/late-Sep disbursements (due ≈ Oct + 30d ≈ Nov 5+) cannot be classified yet; excluding them prevents understating risk. This is exactly the "trap" note from triage.
- Your idea "expected to pay fully before end of quarter" is close but refine: the cutoff is **due_date + 30 ≤ extract date (Nov 5)**, not end of Q3. Loans due in early Oct may already be DPD30-observable even though disbursed in Q3 — include them (cohort = loans *disbursed* in Q3).
- Report two cuts: (a) overall DPD30 rate on observable Q3 cohort, (b) by disbursal month (Jul/Aug fully observable, Sep partial/none) so the reader sees maturity effects.
- Optional extra: credit-loss EGP = Σ unrecovered principal on defaulted loans (ties to README revenue rules).

## Cell 9 — Plus subscribers (stock metric)
- Active as of date X: `plus_start_date <= X` and (`plus_end_date` is null OR `> X`).
- Report at two points: Q3-end (Sep 30) and extract (Nov 5) — shows momentum. Sanity-check `monthly_fee_egp == 49` everywhere; check overlapping/re-subscription rows per user.
- Explicitly state: this is a subscriber COUNT, not summed into deposit volume.

## Cell 10 — KPI summary table (Part 3.1 deliverable core)
One tidy table: Metric | Value | Definition used | Key assumptions/exclusions. All 7 KPIs + headline verdict line ("59M claim: WRONG/CORRECT — true figure X M EGP, gap caused by …").

## Cell 11 — One-page board summary (Part 5.1 deliverable)
Markdown cell, BLUF structure:
1. **Bottom line** (1–2 sentences): headline verdict first.
2. **≤ 3 supporting numbers** — recommend: (a) corrected Q3 deposit volume + gap cause, (b) activation rate, (c) DPD30 rate on matured cohort. Curate ruthlessly; put the rest in an appendix link to the notebook.
3. **One decision ask** for the CPGO — make it specific, e.g. *"Approve using the corrected deposit figure (X M EGP) in the board pack and adopt the definitions appendix as the standing KPI dictionary."*
- Constraint checklist at top of cell: one page ✓, BLUF ✓, ≤3 numbers ✓, single ask ✓.

## Cell 12 — Definitions & assumptions register (markdown)
Dump every stated decision in one place (tz offset/no-DST, dedup rule, test-user list, MAU definition, activation sub-rules, DPD30 observability rule, as-of dates for stock metrics). This is what saves you in the debrief.

---

## Execution order rationale
Cells 1–2 (cleaning/timezone) come **before** every metric so all seven KPIs share one validated base; Cell 3 first among metrics because the headline is the CPGO's actual question; everything else supports the one-pager in Cells 10–11.

## Open questions to resolve while building (answer in notebook, don't guess silently)
- Are the 577 duplicate txn_ids exact copies (dedup safe) or distinct events sharing an id (needs investigation)? ← check in Cell 1
- Do failed deposits or p2p_send inflate anything near 59M? ← bridge table answers this
- Any Q3 loan whose due_date + 30 lands after Nov 5 → excluded from denominator; quantify how many.
